In [40]:
import pandas as pd
import numpy as np


In [41]:
df = pd.read_csv("../data/gene_expression/GSE87213_biofilm_DE_cleaned.csv")
print("Loaded:", df.shape)
df.head()


Loaded: (5983, 21)


,id,gene_symbol,log_FC,log_CPM,deseq.p-value,deseq.adj.pvalue,edger.p.value,edger.adj.p.value,X3225B2.norm,X3225B3.norm,...,WTB3.norm,WTB4.norm,WTB6.norm,X3225B2.raw,X3225B3.raw,X3225B4.raw,WTB3.raw,WTB4.raw,WTB6.raw,log2FoldChange_norm
0,PA14_22460,PA14_22460,4.345,7.733,2.000000e-71,1.200000e-67,4.600000e-80,1.400000e-76,1686,1573,...,99,79,62,1956,1711,1200,122,60,73,4.345
1,PA14_22450,ppiA,4.151,7.630,1.000000e-66,3.000000e-63,2.400000e-85,1.400000e-81,1574,1537,...,84,82,89,1826,1672,1036,104,62,103,4.151
2,PA14_22470,PA14_22470,3.929,9.077,8.800000e-66,1.700000e-62,1.800000e-75,3.500000e-72,4458,3785,...,298,237,269,5171,4117,2938,369,179,313,3.929
3,PA14_22440,bapA,3.773,8.996,3.000000e-62,4.400000e-59,1.100000e-73,1.600000e-70,3877,3928,...,305,293,245,4497,4272,2704,377,222,285,3.773
4,PA14_27070,PA14_27070,3.263,9.016,1.400000e-49,1.600000e-46,2.100000e-53,2.500000e-50,4260,3546,...,339,386,458,4941,3857,2582,421,292,531,3.263


In [42]:
# We only keep the columns needed for engineered features
df_small = df[[
    "gene_symbol",
    "log_FC",            # biofilm logFC
    "deseq.adj.pvalue",  # biofilm padj
    "log2FoldChange_norm"  # planktonic logFC
]].copy()

df_small.rename(columns={
    "log_FC": "biofilm_logFC",
    "deseq.adj.pvalue": "biofilm_padj",
    "log2FoldChange_norm": "planktonic_logFC"
}, inplace=True)

df_small.head()


,gene_symbol,biofilm_logFC,biofilm_padj,planktonic_logFC
0,PA14_22460,4.345,1.200000e-67,4.345
1,ppiA,4.151,3.000000e-63,4.151
2,PA14_22470,3.929,1.700000e-62,3.929
3,bapA,3.773,4.400000e-59,3.773
4,PA14_27070,3.263,1.600000e-46,3.263


In [43]:
df_small["planktonic_padj"] = 0.05


In [44]:
df_small["biofilm_abs_logFC"] = df_small["biofilm_logFC"].abs()
df_small["planktonic_abs_logFC"] = df_small["planktonic_logFC"].abs()


In [45]:
df_small["biofilm_significant"] = (df_small["biofilm_padj"] < 0.05).astype(int)
df_small["planktonic_significant"] = (df_small["planktonic_padj"] < 0.05).astype(int)


In [46]:
# Interaction between logFC values
df_small["logFC_interaction"] = df_small["biofilm_logFC"] * df_small["planktonic_logFC"]

# Interaction between absolute logFC values
df_small["abs_logFC_interaction"] = df_small["biofilm_abs_logFC"] * df_small["planktonic_abs_logFC"]

# Difference between logFC values
df_small["logFC_difference"] = df_small["biofilm_logFC"] - df_small["planktonic_logFC"]

# Difference between absolute logFC values
df_small["abs_logFC_difference"] = df_small["biofilm_abs_logFC"] - df_small["planktonic_abs_logFC"]

# Ratio between logFC values
df_small["logFC_ratio"] = df_small["biofilm_logFC"] / (df_small["planktonic_logFC"] + 1e-6)

# Ratio between absolute logFC values
df_small["abs_logFC_ratio"] = df_small["biofilm_abs_logFC"] / (df_small["planktonic_abs_logFC"] + 1e-6)


In [47]:
df_small["biofilm_label"] = df_small["biofilm_significant"]


In [48]:
output_path = "../data/gene_expression/biofilm_ml_features.csv"
df_small.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", df_small.shape)
df_small.head()


Saved: ../data/gene_expression/biofilm_ml_features.csv
Final shape: (5983, 16)


,gene_symbol,biofilm_logFC,biofilm_padj,planktonic_logFC,planktonic_padj,biofilm_abs_logFC,planktonic_abs_logFC,biofilm_significant,planktonic_significant,logFC_interaction,abs_logFC_interaction,logFC_difference,abs_logFC_difference,logFC_ratio,abs_logFC_ratio,biofilm_label
0,PA14_22460,4.345,1.200000e-67,4.345,0.05,4.345,4.345,1,0,18.879025,18.879025,0.0,0.0,1.0,1.0,1
1,ppiA,4.151,3.000000e-63,4.151,0.05,4.151,4.151,1,0,17.230801,17.230801,0.0,0.0,1.0,1.0,1
2,PA14_22470,3.929,1.700000e-62,3.929,0.05,3.929,3.929,1,0,15.437041,15.437041,0.0,0.0,1.0,1.0,1
3,bapA,3.773,4.400000e-59,3.773,0.05,3.773,3.773,1,0,14.235529,14.235529,0.0,0.0,1.0,1.0,1
4,PA14_27070,3.263,1.600000e-46,3.263,0.05,3.263,3.263,1,0,10.647169,10.647169,0.0,0.0,1.0,1.0,1
